In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai
Raw dir: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/raw
Processed dir: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed


In [2]:
import pandas as pd

from app.infrastructure.pdf_loader import PDFLoader
from app.repositories.document_repository import FileDocumentRepository
from app.services.chunking_service import ChunkingService
from app.services.document_processing_service import DocumentProcessingService
from app.services.embedding_service import EmbeddingService
from app.services.text_cleaning_service import TextCleaningService
from app.services.vectorstore_service import VectorStoreService

In [3]:
pdf_files = sorted((RAW_DIR / "gesetze_im_internet").rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDFs:")
for pdf in pdf_files:
    print("-", pdf.name)

Found 3 PDFs:
- PUEG_RefE_Pflegereform_bf.pdf
- SGB_11.pdf
- SGB_5.pdf


In [4]:
loader = PDFLoader()
repo = FileDocumentRepository()
cleaner = TextCleaningService()
chunker = ChunkingService(chunk_size=1000, chunk_overlap=200)
processor = DocumentProcessingService(cleaning_service=cleaner, chunking_service=chunker)
embedder = EmbeddingService()
vectorstore = VectorStoreService()

/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/app/services/vectorstore_service.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._store = Chroma(


In [5]:
all_chunks = []
records = []

for pdf_path in pdf_files:
    loaded = loader.load(
        str(pdf_path),
        source="gesetze-im-internet",
        document_type="law",
        topic="healthcare-law",
    )
    saved = repo.save(loaded)
    chunks = processor.process(saved)

    all_chunks.extend(chunks)

    records.append(
        {
            "file_name": pdf_path.name,
            "document_id": saved.document_id,
            "chunk_count": len(chunks),
            "first_chunk_preview": chunks[0].text[:250] if chunks else "",
        }
    )

df = pd.DataFrame(records)
df

,file_name,document_id,chunk_count,first_chunk_preview
0,PUEG_RefE_Pflegereform_bf.pdf,d335538b-74d9-4f6f-83ca-cebf17768798,444,Referentenentwurf\ndes Bundesministeriums für ...
1,SGB_11.pdf,0b859b80-8f28-4172-978c-feb589417aa0,981,Ein Service des Bundesministerium der Justiz u...
2,SGB_5.pdf,676aebc2-2db0-46c9-9086-72ed95b36887,3324,Ein Service des Bundesministerium der Justiz u...


In [6]:
sample_text = all_chunks[0].text if all_chunks else "test"
doc_embedding = embedder.embed_documents([sample_text])[0]
query_embedding = embedder.embed_query("Was ist Pflegegrad 3?")

print("Document embedding dimension:", len(doc_embedding))
print("Query embedding dimension:", len(query_embedding))

Document embedding dimension: 1536
Query embedding dimension: 1536


In [7]:
sample_chunks = all_chunks[:20]

vectorstore.add_chunks(sample_chunks)

print(f"Indexed {len(sample_chunks)} chunks")

Indexed 20 chunks


In [8]:
results = vectorstore.similarity_search("Was ist Pflegegrad 3?", top_k=5)

for i, chunk in enumerate(results, start=1):
    print("=" * 80)
    print(i, chunk.document_id, chunk.chunk_index, chunk.metadata.source)
    print(chunk.text[:700])
    print()

1 a045aad7-0d31-46df-8243-289cb921641a 7 gesetze-im-internet
femöglichkeiten zu verbessern. Die Pflegeversicherung stellt hierfür 50 Millionen Euro
pro Jahr zur Verfügung. Voraussetzung ist eine hälftige Kofinanzierung durch das jeweilige
Bundesland bzw. die jeweilige Kommune als für die Sicherstellung der pflegerischen Infra
-
struktur bzw. die Daseinsvorsorge zuständige Beteiligte.
Die Möglichkeiten der Digitalisierung in der Langzeitpflege sollen noch besser genutzt wer-
den. Dazu wird ein Kompetenzzentrum Digitalisierung und Pflege eingerichtet, das die Po-
tentiale zur Verbesserung und Stärkung der pflegerischen Versorgung sowohl für die Be -
troffenen als auch die Pflegenden identifiziert und verbreitet. Das bestehende Förderpro -
gramm nach § 8

2 d335538b-74d9-4f6f-83ca-cebf17768798 7 gesetze-im-internet
femöglichkeiten zu verbessern. Die Pflegeversicherung stellt hierfür 50 Millionen Euro
pro Jahr zur Verfügung. Voraussetzung ist eine hälftige Kofinanzierung durch das jeweilig

In [9]:
output_path = PROCESSED_DIR / "embedding_vectorstore_experiments.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed/embedding_vectorstore_experiments.csv


In [10]:
vectorstore.add_chunks(
    all_chunks,
    batch_size=25
)

results = vectorstore.similarity_search(
    "Was ist Pflegegrad 3?",
    top_k=5,
)

In [11]:
results

[DocumentChunk(document_id='0b859b80-8f28-4172-978c-feb589417aa0', chunk_id='0b859b80-8f28-4172-978c-feb589417aa0-chunk-0880', chunk_index=880, source_path='/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/raw/gesetze_im_internet/SGB_11.pdf', text='ach\nMaßgabe von Satz 3 einem Pflegegrad zugeordnet. Die Zuordnung ist dem Versicherten schriftlich mitzuteilen. Für\ndie Zuordnung gelten die folgenden Kriterien:\n1.\xa0\xa0 Versicherte, bei denen eine Pflegestufe nach den §§ 14 und 15 in der am 31. Dezember 2016 geltenden\nFassung, aber nicht zusätzlich eine erheblich eingeschränkte Alltagskompetenz nach § 45a in der am 31.\nDezember 2016 geltenden Fassung festgestellt wurde, werden übergeleitet\na)\xa0\xa0 von Pflegestufe I in den Pflegegrad 2,\n\nb)\xa0\xa0 von Pflegestufe II in den Pflegegrad 3,\n\nc)\xa0\xa0 von Pflegestufe III in den Pflegegrad 4 oder\n\nd)\xa0\xa0 von Pflegestufe III in den Pflegegrad 5, soweit die Voraussetzungen für Leistungen nach § 36\nAbsatz 4 oder § 43 A